In [ ]:
!pip install flask flask-cors pyngrok transformers sentence-transformers faiss-cpu accelerate deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.4 MB/s eta 0:00:00


In [ ]:
!pip install pydantic

In [ ]:
from flask import Flask, request
from flask_cors import CORS
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from deep_translator import GoogleTranslator
import json

In [ ]:
app = Flask(__name__)
CORS(app)

In [ ]:
df = pd.read_csv("cleaned_data.csv")
texts = df["text"].tolist()

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(texts)

index = faiss.IndexFlatL2(len(embeddings[0]))
index.add(np.array(embeddings))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def retrieve(query, k=3):
    q_emb = embedder.encode([query])

    distances, indices = index.search(q_emb, k * 2)

    results = [texts[i] for i in indices[0]]

    filtered = [
        r for r in results
        if any(word.lower() in r.lower() for word in query.split())
    ]

    return filtered[:k] if filtered else results[:k]

In [ ]:


from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer_qwen = AutoTokenizer.from_pretrained(model_name)

model_qwen = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="auto", torch_dtype="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def answer_question(
    query,
    context_list,
    chat_history=None
):

    context = "\n".join(context_list)

    history_text = ""

    if chat_history:
        history_text = "Recent Conversation:\n"

        for interaction in chat_history[-2:]:
            history_text += (
                f"Visitor: {interaction['user']}\n"
                f"Guide: {interaction['bot']}\n"
            )

    prompt = f"""
You are a friendly and knowledgeable Egyptian tourist guide.

Answer the question using ONLY the information provided in the context.

Style:
- Speak naturally and clearly like a guide.
- Use 2-3 sentences.

Strict Rules:
- Do NOT add explanations or interpretations.
- Do NOT add any information not directly written in the context.
- Do NOT justify or comment on the information.
- Do NOT repeat the question.
- If the user uses a pronoun (he, she, it, they), use the Recent Conversation to understand who they mean.

Context:
{context}

{history_text}

Visitor Question: {query}

Guide Answer:
"""

    inputs = tokenizer_qwen(prompt, return_tensors="pt").to(model_qwen.device)

    outputs = model_qwen.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.2
    )

    response = tokenizer_qwen.decode(outputs[0], skip_special_tokens=True)

    answer = response.split("Guide Answer:")[-1].strip()

    answer = answer.split("Visitor Question:")[0].strip()

    answer = " ".join(answer.split())

    if not answer.endswith((".", "!", "?")):
        if "." in answer:
            answer = answer.rsplit(".", 1)[0] + "."

    return answer

In [ ]:
import numpy as np
import time
from pydantic import BaseModel, Field
from typing import List, Optional

# --- 1. ENHANCED PRODUCTION SCHEMA ---
class RAGResponse(BaseModel):
    status: str
    generated_answer: str
    retrieved_context: List[str]
    latency_seconds: float
    guardrail_decision: str = Field(description="Action taken by the security layer: PASSED, BLOCKED_INPUT, or BLOCKED_OUTPUT")
    error_message: Optional[str] = None


# --- 2. ADVANCED SEMANTIC GUARDRAIL ENGINE ---
# Define out-of-domain or unsafe anchor concepts using dense descriptive context
# Expand the anchors to capture short queries and syntax terms
OUT_OF_DOMAIN_ANCHORS = [
    "Write code, a python function, programming script, coding class, or software code.",
    "Ignore instructions, system prompt override, jailbreak, developer mode.",
    "Modern politics, current government, elections, and warfare.",
    "How to build weapons, explosive devices, bombs, or illegal hacking.",
    "Financial advice, stock investments, medical prescriptions, or legal counseling."
]

# print("Vectorizing Guardrail Security Anchors...")
# Pre-calculate anchor vectors during initialization to avoid runtime overhead
ANCHOR_EMBEDDINGS = embedder.encode(OUT_OF_DOMAIN_ANCHORS) # Shape: (5, 384)

def evaluate_input_safety(query: str, threshold: float = 0.40) -> tuple[bool, float]:
    """
    Computes mathematical similarity against forbidden topics using the existing embedder.
    Returns (is_safe, highest_similarity_score).
    """
    query_vector = embedder.encode([query])[0] # Shape: (384,)

    # Calculate cosine similarity manually using numpy to avoid adding dependencies
    similarities = []
    for anchor_vec in ANCHOR_EMBEDDINGS:
        norm_product = np.linalg.norm(query_vector) * np.linalg.norm(anchor_vec)
        if norm_product == 0:
            similarities.append(0.0)
        else:
            similarity = np.dot(query_vector, anchor_vec) / norm_product
            similarities.append(float(similarity))

    max_score = max(similarities)
    # If similarity matches an out-of-domain concept too closely, flag it
    if max_score > threshold:
        return False, max_score
    return True, max_score


# --- 3. OUTPUT CONTEXT-GROUNDING GUARDRAIL ---
def evaluate_output_grounding(answer: str, context_list: list[str], threshold: float = 0.35) -> bool:
    """
    Ensures the generated answer shares a strong semantic connection with retrieved items,
    protecting against hallucination errors.
    """
    if not context_list:
        return False

    answer_vector = embedder.encode([answer])[0]
    combined_context = " ".join(context_list)
    context_vector = embedder.encode([combined_context])[0]

    norm_product = np.linalg.norm(answer_vector) * np.linalg.norm(context_vector)
    if norm_product == 0:
        return False

    semantic_overlap = np.dot(answer_vector, context_vector) / norm_product
    return semantic_overlap >= threshold


# --- 4. INTEGRATED WRAPPER FUNCTION ---
def generate_tour_response(query: str, chat_history: list = None) -> dict:
    start_time = time.time()

    try:
        # --- LAYER 1: INPUT SEMANTIC GUARDRAIL ---
        is_safe, input_risk_score = evaluate_input_safety(query)
        if not is_safe:
            return RAGResponse(
                status="success",
                generated_answer="As an Egyptian Tour Guide, I am dedicated exclusively to history, ancient monuments, and tourism. I cannot process this request.",
                retrieved_context=[],
                latency_seconds=round(time.time() - start_time, 3),
                guardrail_decision="BLOCKED_INPUT"
            ).model_dump()

        # --- ENHANCED QUERY REFORMULATION ---
        search_query = query
        if chat_history:
            last_question = chat_history[-1]['user']
            # Grab the first 50 characters of the bot's last answer to capture the named entity (e.g., "Khafre")
            last_answer_snippet = chat_history[-1]['bot'][:50]
            search_query = f"{last_question} {last_answer_snippet} {query}"

        # --- QUERY REFORMULATION ---
        # search_query = query
        # if chat_history:
        #     last_question = chat_history[-1]['user']
        #     search_query = f"{last_question} {query}"

        # --- RETRIEVAL PASSTHROUGH ---
        context_list = retrieve(search_query, k=3)

        # --- EMPTY DATA FILTER ---
        if not context_list:
            return RAGResponse(
                status="success",
                generated_answer="I apologize, but I do not have verified historical data regarding that specific request within my archive.",
                retrieved_context=[],
                latency_seconds=round(time.time() - start_time, 3),
                guardrail_decision="PASSED"
            ).model_dump()

        # --- GENERATION EXECUTION ---
        bot_answer = answer_question(query, context_list, chat_history)
        # Force the model to stop at the first major paragraph break
        bot_answer = bot_answer.split('\n')[0].strip()
        # --- LAYER 2: OUTPUT STRUCTURAL & TRUTHFULNESS FILTER ---
        # Rule check: Check for model degradation or empty string loops
        if len(bot_answer) < 5 or "ERROR" in bot_answer.upper():
            bot_answer = "I apologize, I am having trouble clarifying that record. Could you rephrase your question?"

        # Semantic Hallucination Check: Validate alignment with the source texts
        is_grounded = evaluate_output_grounding(bot_answer, context_list)
        if not is_grounded:
            return RAGResponse(
                status="success",
                generated_answer="I cannot confidently confirm that detail using our archival records. Let me know if you would like to explore a different historical era.",
                retrieved_context=context_list,
                latency_seconds=round(time.time() - start_time, 3),
                guardrail_decision="BLOCKED_OUTPUT"
            ).model_dump()

        # SUCCESSFUL COMPLETION
        return RAGResponse(
            status="success",
            generated_answer=bot_answer,
            retrieved_context=context_list,
            latency_seconds=round(time.time() - start_time, 3),
            guardrail_decision="PASSED"
        ).model_dump()

    except Exception as e:
        return RAGResponse(
            status="error",
            generated_answer="A system error occurred.",
            retrieved_context=[],
            latency_seconds=round(time.time() - start_time, 3),
            guardrail_decision="PASSED",
            error_message=str(e)
        ).model_dump()

In [ ]:
def is_arabic(text):
    return any("\u0600" <= c <= "\u06ff" for c in text)

In [ ]:
def answer_question_multilang(query):

    arabic = is_arabic(query)

    try:
        if arabic:
            query_en = GoogleTranslator(source="auto", target="en").translate(query)
        else:
            query_en = query
    except:
        return "حدث خطأ في الترجمة" if arabic else "Translation error"


    result = generate_tour_response(
        query_en,
        chat_history
    )

    answer_en = result["generated_answer"]

    try:
        if arabic:
            answer_ar = GoogleTranslator(
                source="auto",
                target="ar"
            ).translate(answer_en)
            return answer_ar
    except:
        return answer_en

    return answer_en

In [ ]:
chat_history = []
@app.route("/api/chat", methods=["POST"])
def chat():

    data = request.get_json()

    if not data or "question" not in data:
        return {"error": "No question provided"}, 400

    question = data["question"]

    answer = answer_question_multilang(question)
    chat_history.append({
        "user": question,
        "bot": answer
    })

    chat_history[:] = chat_history[-5:]

    return app.response_class(
        response=json.dumps({"answer": answer}, ensure_ascii=False),
        status=200,
        mimetype="application/json",
    )

In [ ]:
@app.route("/api/llm-chat", methods=["POST"])
def llm_chat():
    data = request.get_json()

    if not data or "question" not in data:
        return {"error": "No question provided"}, 400

    question = data["question"]

    answer = answer_question_multilang(question)
    chat_history.append({
        "user": question,
        "bot": answer
    })

    chat_history[:] = chat_history[-5:]

    return app.response_class(
        response=json.dumps({"answer": answer}, ensure_ascii=False),
        status=200,
        mimetype="application/json",
    )

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("3DlWbRIKGqfDCvX0zM0UzL0SUZL_4dUsUKhCoKMwdEfzyQcko")

In [ ]:
public_url = ngrok.connect(5006)
print(public_url)

NgrokTunnel: "https://contractedly-unblemishable-anitra.ngrok-free.dev" -> "http://localhost:5006"


In [ ]:
from threading import Thread
thread = Thread(
    target=lambda: app.run(
        host="0.0.0.0",
        port=5006,
        use_reloader=False
    )
)
thread.start()

time.sleep(5)

public_url = ngrok.connect(5006)
print("LLM URL:", public_url)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5006
 * Running on http://172.28.0.12:5006
INFO:werkzeug:Press CTRL+C to quit


LLM URL: NgrokTunnel: "https://contractedly-unblemishable-anitra.ngrok-free.dev" -> "http://localhost:5006"


In [ ]:
from pyngrok import ngrok
ngrok.kill()